## 1.1 短变量声明
Go 提供了两种声明变量的方式：完整声明和短声明。
前者使用`var`，显式写出类型：

In [1]:
var name string = "bbw"

而短变量声明使用`:=`，Go 会自动推断类型：

In [2]:
name := "BBW"

Cell[2]: Line 1 /tmp/gonb_ad23010e/main.go:3:1: expected declaration, found name package main

 name := "BBW"

ERROR: parsing go files in TempDir "/tmp/gonb_ad23010e": /tmp/gonb_ad23010e/main.go:3:1: expected declaration, found name

上面的运行错误就是，短声明的左边必须至少有一个 **新变量**。如果左边所有变量都已经声明过，代码不会编译通过。

一般而言，短声明放在函数内部使用，不能用于包级别（函数外部）的变量，这里必须用`var`。
## 1.2 多返回值
Go 的函数可以返回多个值，但最常见的用法是一个返回值加一个 **错误值**：

In [9]:
package main

import (
    "errors"
    "fmt"
)

func divide(a, b int) (int, error) {
    if b == 0 {
        return 0, errors.New("division by zero")
    }
    return a / b, nil
}

func main() {
    result, err := divide(10, 2)
    if err != nil {
        fmt.Println("Error:", err)
    } else {
        fmt.Println("Result:", result)
    }
    
    result2, err2 := divide(10, 0)
    if err2 != nil {
        fmt.Println("Error:", err2)
    } else {
        fmt.Println("Result:", result2)
    }
}

Result: 5
Error: division by zero


如果只关心部分返回值，可以用下划线 _ 忽略其他值。和 Python 类似。

In [4]:
func getCoordinates() (x, y, z int) {
    return 10, 20, 30
}

func main() {
    // 只取 x 和 z，忽略 y
    x, _, z := getCoordinates()
    fmt.Println("x:", x, "z:", z)
}

x: 10 z: 30


## 1.3 可变参数
在 Go 里，可以在参数类型前加 `...` 来表示可变参数。

可变参数在函数内部会被当作切片来处理。

In [11]:
func sum(nums ...int) int {		// 表示这个函数可以接收任意多个 int 类型的参数
	total := 0
	for _, n := range nums {	// nums 类型是`[]int`，可以用range遍历
		total += n
	}
	return total
}
func main() {
// 直接写多个参数
fmt.Println(sum(1, 2, 3)) // 输出: 6
fmt.Println(sum(4, 5))    // 输出: 9
// 传入已有的切片
numbers := []int{6, 7, 8} 
fmt.Println(sum(numbers...)) // 输出: 21
}

6
9
21


可变参数必须放在函数参数列表的最后一位。一个函数只能有一个可变参数。
## 1.4 命名返回值
Go 的函数可以给返回值字段起名字。在函数声明的返回值位置，不仅写类型，还给返回值起一个名字。

命名返回值有几个作用：
- 第一，在函数体内，这些名字会被当作普通变量初始化并可用。
- 第二，一个没有参数的 return 语句会返回这些变量的当前值（这叫“裸返回”或“naked return”）。

命名返回值适合用在短函数里，能提高可读性。但函数较长时，显式 return 值反而更清晰。

In [ ]:
// 普通返回值：函数声明里只写【返回类型】，不写名字
func add(a, b int) int {
    sum := a + b // 我需要自己在函数体内声明一个sum变量
    return sum   // return后面必须【显式写出要返回的变量/值】
}

// 命名返回值：函数声明里写【返回值名字 + 类型】
func split(sum int) (x, y int) {
    x = sum * 4 / 9 // 不用再写 var 声明变量，直接赋值即可
    y = sum - x
    return          // return后面可以什么都不写
}

func main() {
    fmt.Println(split(17))
}


7 10


In [ ]:
// 命名返回值 + 裸返回
func rectangleArea(length, width float64) (area float64) {
    area = length * width
    // 裸返回就是return后面什么都不写
    return      // Go 会自动把当前所有命名返回值变量的值返回出去
}

// 显式返回值（更常见）
func rectanglePerimeter(length, width float64) float64 {
    return 2 * (length + width)
}

func main() {
    fmt.Println("Area:", rectangleArea(5, 3))
    fmt.Println("Perimeter:", rectanglePerimeter(5, 3))
}

Area: 15
Perimeter: 16


## 1.5 defer 延迟执行
defer 语句把函数调用推迟到包含它的函数返回之前执行。

In [ ]:
package main

import "fmt"

func main() {
    defer fmt.Println("world")
    fmt.Println("hello")
}

hello
world


defer 延迟的是函数调用，而不是普通语句。也就是说，只有函数可以放在 defer 后面。

多个 defer 语句按后进先出的顺序执行（栈的顺序）。

In [16]:
package main

import "fmt"

func main() {
    fmt.Println("start")
    
    defer fmt.Println("1")
    defer fmt.Println("2")
    defer fmt.Println("3")
    
    fmt.Println("end")
}

start
end
3
2
1


最常见的用途是资源清理：打开文件后立刻 defer f.Close()，加锁后立刻 defer mu.Unlock()。

In [ ]:
func readFile(filename string) error {
    f, err := os.Open(filename)
    if err != nil {
        return err
    }
    defer f.Close()  // 确保函数退出时关闭文件
    
    // 读取文件内容...
    fmt.Println("Reading file:", filename)
    
    return nil
}

defer 后面函数的参数会在 defer 语句执行时立即求值，而不是等到函数真正执行时才求值。

In [ ]:
package main

import "fmt"

func main() {
    i := 0
    defer fmt.Println(i)  // 在声明时就已经计算好参数是 0
	// 虽然函数结束时才执行 fmt.Println，但它打印的是旧值 0，不是 i 的最终值 1
    
    i++
    fmt.Println(i)
}

1
0


## 1.6 select 多路复用
### 1.6.1 并发基础
平时用电脑，打开一个浏览器是一个程序，打开一个微信是另一个程序。每个程序内部，还可以同时干好几个事：比如微信可以一边给你发消息，一边下载别人发的文件，一边刷新朋友圈。

在 Go 里，每一个可以同时独立运行的 “小任务”，就叫一个goroutine。
它最大的特点就是特别轻量：你开几千个、几万个 goroutine 都不会卡，这是 Go 天生的超能力。

只要在一个函数前面加一个go关键字，Go 就会在后台启动一个新的 goroutine 来运行这个函数，原来的代码会继续往下跑，不会等这个函数执行完。
```go
go 函数名(参数)
```

In [ ]:
package main
import (
    "fmt"
    "time"
)

// 一个打印数字的函数
func printNumbers() {
    for i := 1; i <= 3; i++ {
        fmt.Println(i)
        time.Sleep(500 * time.Millisecond) // 睡0.5秒
    }
}

func main() {
    fmt.Println("主程序开始")
    
    // 启动一个后台goroutine运行printNumbers
    go printNumbers()
    
    // 主程序继续往下跑，不会等printNumbers执行完
    fmt.Println("主程序继续做自己的事")
    
    // 让主程序睡2秒，等后台goroutine跑完
    // 如果主程序先退出，所有后台goroutine都会被直接杀死
    time.Sleep(2 * time.Second)
    fmt.Println("主程序结束")
}

### 1.6.2 channel
现在有个问题：两个 goroutine 是独立运行的，它们之间怎么互相传数据呢？
比如 A goroutine 计算出了一个结果，怎么告诉 B goroutine？
- 一个 goroutine 可以把数据（快递）放进快递柜（Channel）
- 另一个 goroutine 可以从同一个快递柜里把数据拿出来
- 一个快递柜同一时间只能放一个东西
- 放东西的人：放完之后会一直卡在那，直到有人把东西拿走（即 **阻塞** ）
- 拿东西的人：如果柜子是空的，会一直卡在那，直到有人放东西进去

In [ ]:
package main
import "fmt"

func main() {
    // 1. 创建一个只能放int类型的快递柜ch
    ch := make(chan int)
		// make：Go 里用来创建复杂类型的关键字（channel、切片、map 都用 make 创建）
		// chan：channel 的关键字，固定写法
		// int：这个 channel（快递柜）只能放int类型的数据，放其他类型会报错
		// ch：这个 channel 的名字，你可以随便起
    
    // 2. 启动一个后台goroutine
    go func() {
        fmt.Println("后台goroutine：开始计算")
        result := 10 + 20
        fmt.Println("后台goroutine：计算完成，结果是", result)
        
        // 3. 把计算结果放进ch快递柜
        // 如果没人来拿，这里会一直卡住
        ch <- result
        fmt.Println("后台goroutine：数据已经发出去了")
    }()
    
    fmt.Println("主程序：等待后台结果...")
    
    // 4. 从ch快递柜里拿结果
    // 如果没人放东西，这里会一直卡住
    val := <-ch
		// 语法确实奇怪，又要声明又要箭头
    
    fmt.Println("主程序：收到结果了，是", val)
}

### 1.6.3 select
- goroutine 是后台任务
- channel 是 goroutine 之间传数据的快递柜
- 从空 channel 拿东西会卡住，往满 channel 放东西会卡住
那现在有个新问题：如果我同时有好几个快递柜，我想同时等着它们，哪个柜子先有东西，我就先拿哪个，该怎么办？

这就是select诞生的目的！
- 它会一直盯着所有柜子
- 只要有任何一个柜子可以放东西或者可以拿东西，它就立刻去做那个操作
- 如果多个柜子同时就绪，它会随机选一个（没有先后顺序）
- 如果所有柜子都没就绪，它会一直卡在那等（除非有 default 分支）

```
select {
case 操作1:
    操作1完成后执行的代码
case 操作2:
    操作2完成后执行的代码
default:
    所有操作都没就绪时执行的代码
}
```
> 和C的Switch case不同，select 的 case 里，只能写 channel 的发送或接收操作，不能写别的

In [18]:
package main
import (
    "fmt"
    "time"
)

func main() {
    // 创建两个空的字符串快递柜ch1和ch2
    ch1 := make(chan string)
    ch2 := make(chan string)
    
    // 启动第一个后台goroutine
    go func() {
        time.Sleep(1 * time.Second) // 睡1秒，模拟干活
        ch1 <- "from ch1" // 1秒后，把"from ch1"放进ch1
    }()
    
    // 启动第二个后台goroutine
    go func() {
        time.Sleep(2 * time.Second) // 睡2秒，模拟干活
        ch2 <- "from ch2" // 2秒后，把"from ch2"放进ch2
    }()
    
    // 循环2次，因为我们要拿两个快递
    for i := 0; i < 2; i++ {
        // 进入select：同时守着ch1和ch2两个快递柜
        select {
        // 情况1：ch1有东西可以拿了
        // 把拿出来的东西存到msg1，然后打印
        case msg1 := <-ch1:
            fmt.Println(msg1)
        // 情况2：ch2有东西可以拿了
        // 把拿出来的东西存到msg2，然后打印
        case msg2 := <-ch2:
            fmt.Println(msg2)
        }
    }
}

from ch1
from ch2


默认的 select 会一直阻塞，但加上default分支后，它就变成了非阻塞的：如果所有 channel 都没就绪，直接执行 default，不会卡住。

In [19]:
package main
import "fmt"

func main() {
    // 创建一个空的int快递柜ch，没有任何goroutine往里面放东西
    ch := make(chan int)
    
    select {
    // 尝试从ch拿东西：永远不会成功，因为没人放
    case val := <-ch:
        fmt.Println("Received:", val)
    // 所有case都没就绪，直接执行这里，不会卡住
    default:
        fmt.Println("No data available, continuing")
    }
}

No data available, continuing


select 配合 time.After 可以实现超时控制。

In [20]:
package main
import (
    "fmt"
    "time"
)

func main() {
    ch := make(chan string)
    
    // 启动一个后台goroutine，模拟一个很慢的接口
    go func() {
        time.Sleep(2 * time.Second) // 这个接口需要2秒才能返回结果
        ch <- "result"
    }()
    
    select {
    // 情况1：1秒内从ch拿到了接口返回的结果
    case res := <-ch:
        fmt.Println("接口调用成功，结果：", res)
    // 情况2：1秒到了，还没拿到结果，触发超时
    case <-time.After(1 * time.Second):
        fmt.Println("超时了！接口太慢了")
    }
}

超时了！接口太慢了


- time.After(1 * time.Second)是 Go 标准库的函数，它会返回一个新的 channel
- 当 1 秒时间到了之后，这个 channel 会自动被放入一个值（也就是“就绪”了）
- 所以这里的 select 其实是在同时等两个事件：
    - 接口返回结果（ch 就绪）
    - 超时时间到了（time.After 返回的 channel 就绪）
哪个先发生，就执行哪个 case。

## 1.7 类型断言
C 是 **纯静态类型**：一个变量声明成int，它就永远只能存整数；声明成char*，就永远只能存字符串指针，类型从编译那一刻起就固定死了，所以根本不需要 **类型断言** 这种东西。

而 Go 有一个 C 完全没有的核心概念：接口（interface）。类型断言和类型开关，就是专门为接口设计的语法。
### 1.7.1 接口前置
在讲类型断言之前，必须先理解一个最关键的事实：
> **接口类型的变量，可以存任何类型的值**。

打个最通俗的比方：
- 普通变量（比如`int a`、`string s`）就像**贴了标签的透明玻璃瓶**：标签写着“只能装水”，你就只能装水；标签写着“只能装可乐”，就只能装可乐。而且瓶子是透明的，你一眼就能看到里面装的是什么。
- 接口类型的变量就像**不透明的黑盒子**：盒子外面只写着“可以装任何东西”，但你看不到里面到底装的是什么。
```go
// 一个空接口变量i，可以装任何东西
var i interface{}

i = 10          // 装整数
i = "hello"     // 装字符串
i = true        // 装布尔值
i = 3.14        // 装浮点数
i = []int{1,2,3} // 装切片
```
接口变量看起来是一个变量，但它内部其实存了**两个东西**：
1. **类型信息**：里面装的那个值的实际类型（比如`int`、`string`）
2. **值信息**：里面装的那个值本身（比如`10`、`"hello"`）

比如`var i interface{} = "hello"`，这个`i`内部存的就是：`(类型: string, 值: "hello")`。

**类型断言的本质，就是把这两个东西从黑盒子里拿出来**。

### 1.7.2 类型断言
类型断言有两种写法：不安全的写法和安全的写法。
#### 不安全的写法
```
值 := 接口变量.(具体类型)
```
- `接口变量`：必须是一个接口类型的变量（比如上面的`i`）
- `.(具体类型)`：这是固定写法，**中间没有空格**！意思是“我断言这个接口变量里面装的是这个具体类型”
- 如果断言对了：就把里面的值取出来，赋值给左边的变量
- 如果断言错了：程序会直接**崩溃（panic）**
#### 安全的写法
```
值, ok := 接口变量.(具体类型)
```
- 多了一个返回值`ok`，是布尔类型
- 如果断言对了：`ok = true`，`值`是里面的具体值
- 如果断言错了：`ok = false`，`值`是该类型的**零值**（比如int是0，string是空字符串），程序不会崩溃


In [23]:
package main
import "fmt"

func main() {
    var i interface{} = "hello"
    
    // 不安全的断言：我赌i里面是int
    // 但i里面明明是string！断言失败
    // 程序会直接崩溃，抛出panic，下面的代码永远不会执行
    n := i.(int)
    fmt.Println(n)
}

panic: interface conversion: interface {} is string, not int

goroutine 1 [running]:
main.main()
	 [[ Cell [23] Line 10 ]] /tmp/gonb_ad23010e/main.go:54 +0x28
exit status 2


Cell[23]: Line 10 /tmp/gonb_ad23010e/main.go:54 +0x28 // 不安全的断言：我赌i里面是int
 // 但i里面明明是string！断言失败
 // 程序会直接崩溃，抛出panic，下面的代码永远不会执行
 n := i.(int)
 fmt.Println(n)
}

ERROR: exit status 2

In [22]:
package main
import "fmt"

func main() {
    // 1. 创建一个空接口变量i，里面装了字符串"hello"
    // 现在i内部是：(类型: string, 值: "hello")
    var i interface{} = "hello"
    
    // 2. 不安全的断言：我赌i里面是string
    // 断言正确，所以s = "hello"
    s := i.(string)
    fmt.Println(s) // 输出 hello
    
    // 3. 安全的断言：检查i里面是不是string
    // 正确，所以s2 = "hello"，ok = true
    s2, ok := i.(string)
    fmt.Println(s2, ok) // 输出 hello true
    
    // 4. 安全的断言：检查i里面是不是int
    // 错误！i里面是string，不是int
    // 所以n = int的零值0，ok = false
    n, ok := i.(int)
    fmt.Println(n, ok) // 输出 0 false
}

hello
hello true
0 false


### 1.7.3 类型开关
现在又有一个新问题：如果我想依次检查这个黑盒子里装的是什么类型，然后根据不同的类型做不同的处理，该怎么办？

比如：如果是int，就打印“这是一个整数”；如果是string，就打印“这是一个字符串”；如果是bool，就打印“这是一个布尔值”。

你当然可以用多个`if ok`来写：
```go
if v, ok := i.(int); ok {
    fmt.Printf("int: %d\n", v)
} else if v, ok := i.(string); ok {
    fmt.Printf("string: %s\n", v)
} else if v, ok := i.(bool); ok {
    fmt.Printf("bool: %t\n", v)
} else {
    fmt.Printf("unknown type\n")
}
```
但 Go 专门设计了 **类型开关（type switch）** 来简化这种写法。
```go
switch 变量 := 接口变量.(type) {
case 类型1:
    // 变量是类型1时执行
case 类型2:
    // 变量是类型2时执行
default:
    // 都不匹配时执行
}
```

In [24]:
package main
import "fmt"

// 函数describe接收一个空接口类型的参数i
// 意思是：这个函数可以接收任何类型的参数
func describe(i interface{}) {
    // 类型开关：检查i里面的实际类型
    // v会自动变成对应case的具体类型
    switch v := i.(type) {
    case int:
        // 进入这个case时，v就是int类型
        fmt.Printf("int: %d\n", v)
    case string:
        // 进入这个case时，v就是string类型
        fmt.Printf("string: %s\n", v)
    case bool:
        // 进入这个case时，v就是bool类型
        fmt.Printf("bool: %t\n", v)
    default:
        // 都不匹配时进入这里
        // 此时v还是空接口类型
        fmt.Printf("unknown type: %T\n", v)
    }
}

func main() {
    describe(42)     // 输出 int: 42
    describe("hello")// 输出 string: hello
    describe(true)   // 输出 bool: true
    describe(3.14)   // 输出 unknown type: float64
}

int: 42
string: hello
bool: true
unknown type: float64


## 1.8 for range
### 1.8.1 Python range vs Go range
| 对比维度 | Python `range()` | Go `range` |
|---------|-----------------|-----------|
| **核心功能** | **生成一个数字序列**：本身不遍历任何东西，只是生成一串连续的整数 | **遍历一个已经存在的数据结构**：永远是“遍历某个东西”，不会凭空生成数字 |
| **用法示例** | `for i in range(4):` → 生成0,1,2,3四个数字 | `for i, v := range nums:` → 遍历nums这个切片 |
| **返回值** | 只返回数字本身 | 根据遍历的东西不同，返回1个或2个值（索引/键 + 值） |

- Python的`range`是**生产者**：它自己造出数字给你用，类似的定位是`enumerate`
- Go的`range`是**遍历器**：它只会遍历你已经有的东西，不会自己造任何东西

### 1.8.2 通用语法
Go的`range`是专门用来遍历各种数据结构的语法糖，通用格式是：
```go
for 第一个返回值, 第二个返回值 := range 要遍历的东西 {
    // 循环体
}
```
不同数据结构的两个返回值含义不同。
#### 遍历数组 / 切片
这是最常用的用法，遍历数组或切片时，`range`返回两个值：
- 第一个值：元素的**索引**（从0开始）
- 第二个值：元素的**值本身**

In [ ]:
package main
import "fmt"

func main() {
    // 定义一个切片，里面有4个元素
    nums := []int{2, 4, 6, 8}
    
    // 遍历nums这个切片
    // i是索引，v是对应位置的值
    for i, v := range nums {
        fmt.Printf("index: %d, value: %d\n", i, v)
    }
}

// nums = [2, 4, 6, 8]
// # Go的for range遍历切片，完全等价于Python的enumerate
// for i, v in enumerate(nums):
//     print(f"index: {i}, value: {v}")

index: 0, value: 2
index: 1, value: 4
index: 2, value: 6
index: 3, value: 8


#### 遍历 map
> Go 中 map 的遍历顺序是随机的，下面就是我roll到的输出的顺序不一样。
> 而 Python 3.7 + 中，map（字典）的遍历顺序是插入顺序。

In [28]:
package main
import "fmt"

func main() {
    // 定义一个map，键是string，值是int
    m := map[string]int{"a": 1, "b": 2, "c": 3}
    
    // 遍历map
    // k是键，v是对应的值
    for k, v := range m {
        fmt.Printf("key: %s, value: %d\n", k, v)
    }
}

key: b, value: 2
key: c, value: 3
key: a, value: 1


#### 遍历字符串
- 第一个值：这个字符的**起始字节索引**（不是字符的序号！）
- 第二个值：这个字符的**Unicode码点（rune类型）**

In [ ]:
package main
import "fmt"

func main() {
    s := "Hello, 世界"
    
    // 遍历字符串
    // i是字节索引，r是Unicode字符
    for i, r := range s {
        fmt.Printf("index: %d, rune: %c\n", i, r)
    }
}

// s = "Hello, 世界"
// for i, c in enumerate(s):
//     print(f"index: {i}, char: {c}")

index: 0, rune: H
index: 1, rune: e
index: 2, rune: l
index: 3, rune: l
index: 4, rune: o
index: 5, rune: ,
index: 6, rune:  
index: 7, rune: 世
index: 10, rune: 界


不能简单理解成index，因为索引从7直接跳到了10。
**核心原因**：Go的字符串是 **UTF-8编码的字节数组**，不是字符数组。
- 英文字母、数字、符号：占1个字节
- 中文、日文、韩文等：占3个字节
- 表情符号：占4个字节
所以“世”这个字占了3个字节（索引7、8、9），下一个字“界”就从索引10开始了。

> 相比之下python一个`enumerate`走天下

#### 遍历Channel
遍历 channel 时，range只返回一个值：从 channel 里取出的数据。
```go
ch := make(chan int)
// 遍历channel：会一直从ch里拿数据，直到ch被关闭
for v := range ch {
    fmt.Println(v)
}
```
这个 Python 里没有直接对应的东西，简单记住就行：for range ch会一直阻塞等待 channel 的数据，直到有人关闭 channel。
### 1.8.3 下换线
Go有一个非常严格的规定：**声明了的变量必须使用，否则编译错误**。

所以如果你遍历的时候只想要索引，不想要值，不能写`for i, v := range nums`然后不用v，会直接报错。这时候你需要用**下划线`_`**来忽略不需要的返回值。

比如`for _, v := range nums{...}`，python也一样。

## 1.9 结构体与面向接口
很多人会说“Go是面向过程的”，因为它没有`class`、没有`extends`、没有继承这些Java/C++里传统OOP的关键词。
也有人会说“Go是面向对象的”，因为它支持封装、多态这些OOP的核心特性。

**正确的说法是**：Go是一门**“面向接口+组合优先”**的语言。它抛弃了传统OOP里最复杂、最容易出问题的“继承”，用更简单、更灵活的“组合”来实现代码复用；用“接口”来实现多态。

- 传统面向对象（Java）：“我是一个人，所以我会说话、会走路”（继承自人类）
- Go：“我有一个人，所以我会说话、会走路”（组合了人类的能力）
- 纯面向过程（C）：“我写一个说话的函数，谁需要谁调用”
### 1.9.1 结构体标签
结构体标签就是**给结构体字段加的一个“备注”**。这个备注不是给人看的，是给**第三方库**看的。库可以在运行时读取这个备注，然后根据备注的内容做对应的处理。
最常见的用途就是 **JSON序列化**：
- 反序列化（Unmarshal / Deserialize） 是把 JSON 格式的文本（或字节流）转换成 Go 语言中的数
- 对应的，序列化（Marshal / Serialize） 是把 Go 数据转换成 JSON 文本。

In [30]:
package main
import (
    "encoding/json"
    "fmt"
)

// 定义一个User结构体
type User struct {
    // 字段名是Name（Go里大写开头表示公开）
    // 标签`json:"name"`告诉json库：
    // 序列化的时候，这个字段在JSON里的key叫"name"（小写）
    Name  string `json:"name"`
    
    Age   int    `json:"age"`
    
    // omitempty的意思是：如果这个字段的值是零值（空字符串、0、false等）
    // 序列化的时候就不要输出这个字段
    Email string `json:"email,omitempty"`
}

func main() {
    // 创建一个User对象，Email字段没赋值，是空字符串
    u := User{
        Name: "张三",
        Age:  30,
    }
    
    // 把结构体序列化成JSON字节数组
    data, _ := json.Marshal(u)
    // 转换成字符串打印
    fmt.Println(string(data))
}

{"name":"张三","age":30}


Email 字段因为是空字符串，加了omitempty就没有被输出。如果去掉omitempty，输出就会变成`{"name":"张三","age":30,"email":""}`。
### 1.9.2 嵌入与组合
传统OOP用“继承”实现代码复用，Go用“嵌入”实现“组合”，这是两者最大的区别。
#### 结构体嵌入
**把一个结构体放到另一个结构体里，不写变量名，就是嵌入**。

嵌入之后，外层结构体可以 **直接访问** 被嵌入结构体的所有字段和方法，就像这些字段和方法是外层结构体自己的一样。

In [31]:
package main
import "fmt"

// 定义一个Person结构体，有Name和Age两个字段
type Person struct {
    Name string
    Age  int
}

// 给Person定义一个Greet方法
func (p Person) Greet() string {
    return "Hello, I'm " + p.Name
}

// 定义一个Employee结构体
type Employee struct {
    Person      // 这里就是嵌入！没有写变量名，直接写类型名
    Company string
    Salary  int
}

func main() {
    // 创建一个Employee对象
    e := Employee{
        // 嵌入的结构体需要显式初始化
        Person:  Person{Name: "李四", Age: 25},
        Company: "Tech Corp",
        Salary:  50000,
    }
    
    // ✅ 可以直接访问嵌入的Person的字段Name
    // 等价于 e.Person.Name
    fmt.Println(e.Name) // 输出 李四
    
    // ✅ 可以直接调用嵌入的Person的方法Greet
    // 等价于 e.Person.Greet()
    fmt.Println(e.Greet()) // 输出 Hello, I'm 李四
    
    // 当然也可以访问Employee自己的字段
    fmt.Println(e.Company) // 输出 Tech Corp
}

李四
Hello, I'm 李四
Tech Corp


很多人看到上面的代码会说：“这不就是继承吗？Employee继承了Person的字段和方法。”

**大错特错！** 这是组合，不是继承。本质区别是：
- 继承是“is-a”关系：Employee **是一个** Person
- 组合是“has-a”关系：Employee **有一个** Person

虽然用起来看起来很像，但组合比继承灵活得多：
1. 你可以嵌入多个类型：比如`type Teacher struct { Person, Employee, TeacherInfo }`，传统继承只能单继承
2. 你可以在运行时替换嵌入的对象：继承是静态的，编译时就确定了
3. 不会有“钻石继承”的问题：传统OOP里最头疼的问题之一
#### 嵌入与接口
嵌入还有一个非常强大的特性：**如果被嵌入的类型实现了某个接口，那么外层结构体也自动实现了这个接口**。

In [ ]:
package main
import "fmt"

// 定义一个Greeter接口，有一个Greet方法
type Greeter interface {
    Greet() string
}

// 定义一个Dog结构体，实现了Greeter接口
type Dog struct {
    Name string
}
func (d Dog) Greet() string {
    return d.Name + " says woof!"
}

// 定义一个Pet结构体，嵌入了Dog
type Pet struct {
    Dog  // 嵌入Dog
    Age  int
}

func main() {
    // 声明一个Greeter接口类型的变量g
    var g Greeter
    
    // ✅ 可以把Pet赋值给g！
    // 因为Pet嵌入了Dog，而Dog实现了Greeter接口
    // 所以Pet自动实现了Greeter接口
    g = Pet{Dog: Dog{Name: "Buddy"}, Age: 3}
    
    fmt.Println(g.Greet()) // 输出 Buddy says woof!
}

Buddy says woof!


## 1.10 包
**包（package）** 本质就是**代码文件夹/代码模块**，作用统一：
1. 把代码分类拆分，不让所有代码堆在一个文件里
2. 区分同名函数/类型，避免命名冲突
3. 实现代码复用

- Java：`package 包名` 声明包，`import 包路径` 导入
- Python：文件夹 + `__init__.py` 构成包，`import 包名`
- Go：每个 `.go` 文件第一行必须写 `package 包名` 声明所属包，同样用 `import` 导入
### 1.10.1 基础导入
```go
// 单行导入
import "fmt"

// 批量导入（推荐写法）
import (
    "fmt"
    "time"
)
```
导入后使用：`包名.内容`，例 `fmt.Println()`、`time.Sleep()`。

假如项目里同时有两个都叫 `fmt` 的包，或者有一个很长的包名，就需要用到包别名/包路径。

```go
package main

import (
    "fmt"               // 标准库 fmt，原名使用
    myfmt "mylib/fmt"   // 给自定义包起别名：myfmt
    // 自定义别名 "包完整路径"
)

func main() {
    fmt.Println("standard fmt")  // 用原生 fmt
    myfmt.Println("custom fmt")  // 用别名调用自定义包
}
```
### 1.10.2 包匿名导入
有些包 **你完全不用它的函数/变量**，只需要：**把这个包加载进来，自动跑它内部的 `init` 函数**。
这时就用**匿名导入**：`import _ "包路径"`
- 下划线 `_` = 匿名别名
- 特点：导入后**不能用 包名.xxx** 调用内容，仅触发包初始化。
```go
package main

import (
    "database/sql"
    // 匿名导入 MySQL 驱动包
    _ "github.com/go-sql-driver/mysql"
)

func main() {
    // 不用写 mysql.xxx，驱动已经被自动注册
    db, err := sql.Open("mysql", "连接地址")
    _ = db
    _ = err
}
```
原理：
1. `mysql` 驱动包内部靠 `init` 函数向标准库 `sql` 注册驱动
2. 我们 **不需要主动调用驱动里任何代码**，只需要让它加载执行 `init`
3. 所以用 `_` 匿名导入，只加载、不使用。

### 1.10.3 init 函数
#### 基础规则
1. 函数名固定：`init`
2. **无参数、无返回值**，写法固定：`func init() {}`
3. **不能手动调用**，程序自动执行
4. 一个包内**可以写多个 init 函数**
5. **包被导入/加载时，自动执行 init**，早于 `main` 函数。

In [34]:
package main

import "fmt"

// 第一个 init
func init() {
    fmt.Println("first init")
}

// 第二个 init
func init() {
    fmt.Println("second init")
}

func main() {
    fmt.Println("main function")
}

second init
main function


多包就逐步执行。